# 台股技術分析工具 Taiwan Stock Technical Analyzer

這個 Notebook 可以：
- 📊 抓取成交量前 50 名的台股
- 📈 計算技術指標（MA、RSI、MACD、KD、布林通道）
- 🎯 應用投資策略篩選符合條件的股票
- 📉 視覺化股票技術圖表

## 1. 安裝與導入必要套件

In [ ]:
# 安裝必要套件（在 Colab 上執行）
!pip install yfinance pandas-ta -q

In [ ]:
# 導入套件
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta as ta
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

# 設定顯示選項
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

# 設定圖表樣式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ 套件導入成功！")

## 2. 定義台股分析器類別

In [ ]:
class TaiwanStockAnalyzer:
    """台股分析器"""

    def __init__(self):
        self.taiwan_stocks = []
        self.stock_data = {}

    def get_taiwan_stock_list(self):
        """獲取台股列表（前50大市值公司範例）"""
        common_stocks = [
            '2330.TW', '2317.TW', '2454.TW', '2308.TW', '2882.TW',
            '2881.TW', '2886.TW', '2891.TW', '2412.TW', '2303.TW',
            '2002.TW', '1301.TW', '1303.TW', '2884.TW', '2892.TW',
            '2887.TW', '2880.TW', '2885.TW', '2883.TW', '2890.TW',
            '2207.TW', '2382.TW', '2357.TW', '2409.TW', '3008.TW',
            '2327.TW', '2379.TW', '2301.TW', '3711.TW', '2105.TW',
            '2912.TW', '2395.TW', '5880.TW', '2324.TW', '2345.TW',
            '2408.TW', '6505.TW', '1216.TW', '2603.TW', '2609.TW',
            '2610.TW', '3045.TW', '4904.TW', '2356.TW', '2377.TW',
            '3034.TW', '2474.TW', '6415.TW', '5269.TW', '3231.TW',
        ]
        return common_stocks

    def fetch_stock_data(self, stock_code, period='3mo', interval='1d'):
        """抓取單一股票數據"""
        try:
            stock = yf.Ticker(stock_code)
            df = stock.history(period=period, interval=interval)

            if df.empty:
                print(f"⚠️  {stock_code} 無數據")
                return None

            info = stock.info
            stock_name = info.get('longName', info.get('shortName', stock_code))
            print(f"✅ {stock_code} ({stock_name}) 數據獲取成功，共 {len(df)} 筆")
            return df

        except Exception as e:
            print(f"❌ {stock_code} 數據獲取失敗: {e}")
            return None

    def calculate_technical_indicators(self, df):
        """計算技術指標"""
        if df is None or df.empty:
            return None

        df = df.copy()

        # 移動平均線
        df['MA5'] = ta.sma(df['Close'], length=5)
        df['MA10'] = ta.sma(df['Close'], length=10)
        df['MA20'] = ta.sma(df['Close'], length=20)
        df['MA60'] = ta.sma(df['Close'], length=60)

        # RSI
        df['RSI'] = ta.rsi(df['Close'], length=14)

        # MACD
        macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
        if macd is not None:
            df['MACD'] = macd['MACD_12_26_9']
            df['MACD_signal'] = macd['MACDs_12_26_9']
            df['MACD_histogram'] = macd['MACDh_12_26_9']

        # KD 隨機指標
        stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=9, d=3, smooth_k=3)
        if stoch is not None:
            df['K'] = stoch['STOCHk_9_3_3']
            df['D'] = stoch['STOCHd_9_3_3']

        # 布林通道
        bbands = ta.bbands(df['Close'], length=20, std=2)
        if bbands is not None:
            df['BB_upper'] = bbands['BBU_20_2.0']
            df['BB_middle'] = bbands['BBM_20_2.0']
            df['BB_lower'] = bbands['BBL_20_2.0']

        return df

    def get_top_volume_stocks(self, stock_list=None, top_n=50):
        """獲取成交量前 N 名的股票"""
        if stock_list is None:
            stock_list = self.get_taiwan_stock_list()

        print(f"📊 正在獲取 {len(stock_list)} 支股票的成交量數據...\n")

        volume_data = []

        for i, stock_code in enumerate(stock_list, 1):
            try:
                stock = yf.Ticker(stock_code)
                hist = stock.history(period='2d')

                if not hist.empty:
                    latest_volume = hist['Volume'].iloc[-1]
                    info = stock.info
                    stock_name = info.get('longName', info.get('shortName', stock_code))

                    volume_data.append({
                        'code': stock_code,
                        'name': stock_name,
                        'volume': latest_volume,
                    })

                    print(f"  [{i}/{len(stock_list)}] {stock_code} ({stock_name}): {latest_volume:,.0f}")

                if i % 10 == 0:
                    time.sleep(1)

            except Exception as e:
                print(f"  [{i}/{len(stock_list)}] {stock_code} 獲取失敗: {e}")

        volume_df = pd.DataFrame(volume_data)
        volume_df = volume_df.sort_values('volume', ascending=False).head(top_n)

        print(f"\n✅ 成交量前 {top_n} 名股票獲取完成！\n")
        return volume_df

    def analyze_stocks(self, stock_list, period='3mo'):
        """批量分析股票並計算技術指標"""
        if isinstance(stock_list, pd.DataFrame):
            stock_codes = stock_list['code'].tolist()
        else:
            stock_codes = stock_list

        print(f"🔍 開始分析 {len(stock_codes)} 支股票...\n")

        results = {}

        for i, stock_code in enumerate(stock_codes, 1):
            print(f"[{i}/{len(stock_codes)}] 分析 {stock_code}...")
            df = self.fetch_stock_data(stock_code, period=period)

            if df is not None:
                df_with_indicators = self.calculate_technical_indicators(df)
                results[stock_code] = df_with_indicators

            if i % 5 == 0:
                time.sleep(1)

        print(f"\n✅ 分析完成！成功分析 {len(results)} 支股票\n")
        return results

    def display_analysis_summary(self, stock_data_dict, stock_info_df=None):
        """顯示分析摘要"""
        summary = []

        for stock_code, df in stock_data_dict.items():
            if df is None or df.empty:
                continue

            latest = df.iloc[-1]

            stock_name = stock_code
            if stock_info_df is not None:
                match = stock_info_df[stock_info_df['code'] == stock_code]
                if not match.empty:
                    stock_name = f"{match.iloc[0]['name']} ({stock_code})"

            summary.append({
                '股票': stock_name,
                '收盤價': f"{latest['Close']:.2f}",
                'MA5': f"{latest['MA5']:.2f}" if pd.notna(latest.get('MA5')) else 'N/A',
                'MA20': f"{latest['MA20']:.2f}" if pd.notna(latest.get('MA20')) else 'N/A',
                'RSI': f"{latest['RSI']:.2f}" if pd.notna(latest.get('RSI')) else 'N/A',
                'K': f"{latest['K']:.2f}" if pd.notna(latest.get('K')) else 'N/A',
                'D': f"{latest['D']:.2f}" if pd.notna(latest.get('D')) else 'N/A',
                '成交量': f"{latest['Volume']:,.0f}",
            })

        summary_df = pd.DataFrame(summary)
        print("📈 技術指標摘要\n")
        display(summary_df)
        return summary_df

    def plot_stock_chart(self, stock_code, df, figsize=(14, 10)):
        """繪製股票技術分析圖表"""
        if df is None or df.empty:
            print(f"無法繪製 {stock_code} 的圖表：數據為空")
            return

        fig, axes = plt.subplots(4, 1, figsize=figsize, sharex=True)

        # 價格與移動平均線
        axes[0].plot(df.index, df['Close'], label='收盤價', linewidth=2)
        axes[0].plot(df.index, df['MA5'], label='MA5', alpha=0.7)
        axes[0].plot(df.index, df['MA20'], label='MA20', alpha=0.7)
        axes[0].plot(df.index, df['MA60'], label='MA60', alpha=0.7)
        axes[0].set_ylabel('價格')
        axes[0].set_title(f'{stock_code} 技術分析圖表', fontsize=14, fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # 成交量
        axes[1].bar(df.index, df['Volume'], alpha=0.5, color='steelblue')
        axes[1].set_ylabel('成交量')
        axes[1].grid(True, alpha=0.3)

        # RSI
        if 'RSI' in df.columns:
            axes[2].plot(df.index, df['RSI'], label='RSI', color='purple', linewidth=2)
            axes[2].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='超買')
            axes[2].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='超賣')
            axes[2].set_ylabel('RSI')
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)

        # MACD
        if 'MACD' in df.columns:
            axes[3].plot(df.index, df['MACD'], label='MACD', color='blue', linewidth=2)
            axes[3].plot(df.index, df['MACD_signal'], label='Signal', color='red', linewidth=2)
            axes[3].bar(df.index, df['MACD_histogram'], label='Histogram', alpha=0.3, color='gray')
            axes[3].set_ylabel('MACD')
            axes[3].legend()
            axes[3].grid(True, alpha=0.3)

        plt.xlabel('日期')
        plt.tight_layout()
        plt.show()


print("✅ TaiwanStockAnalyzer 類別定義完成！")

## 3. 定義投資策略

In [ ]:
class TradingStrategies:
    """交易策略類別"""

    def golden_cross(self, df):
        """黃金交叉：MA5 向上突破 MA20"""
        if len(df) < 2:
            return False
        current = df.iloc[-1]
        previous = df.iloc[-2]
        return (current['MA5'] > current['MA20'] and previous['MA5'] <= previous['MA20'])

    def rsi_oversold(self, df):
        """RSI 超賣：RSI < 30"""
        if len(df) < 1:
            return False
        return df.iloc[-1]['RSI'] < 30

    def rsi_overbought(self, df):
        """RSI 超買：RSI > 70"""
        if len(df) < 1:
            return False
        return df.iloc[-1]['RSI'] > 70

    def volume_surge(self, df):
        """量增價漲：成交量放大且股價上漲"""
        if len(df) < 21:
            return False
        current = df.iloc[-1]
        previous = df.iloc[-2]
        avg_volume = df['Volume'].iloc[-20:].mean()
        return (current['Volume'] > avg_volume * 1.5 and current['Close'] > previous['Close'])

    def kd_golden_cross(self, df):
        """KD 黃金交叉：K 線向上突破 D 線"""
        if len(df) < 2:
            return False
        current = df.iloc[-1]
        previous = df.iloc[-2]
        return (current['K'] > current['D'] and previous['K'] <= previous['D'])

    def macd_bullish(self, df):
        """MACD 多頭：MACD 柱狀圖由負轉正"""
        if len(df) < 2:
            return False
        current = df.iloc[-1]
        previous = df.iloc[-2]
        return (current['MACD_histogram'] > 0 and previous['MACD_histogram'] <= 0)

    def combined_bullish(self, df):
        """綜合多頭：MA5>MA20、RSI 40-70、MACD>0、成交量放大"""
        if len(df) < 20:
            return False
        current = df.iloc[-1]
        avg_volume = df['Volume'].iloc[-20:].mean()
        return (current['MA5'] > current['MA20'] and
                40 < current['RSI'] < 70 and
                current['MACD_histogram'] > 0 and
                current['Volume'] > avg_volume)


def apply_strategy(stock_data_dict, strategy_name, stock_info_df=None):
    """應用投資策略"""
    strategies = TradingStrategies()
    strategy_methods = {
        'golden_cross': strategies.golden_cross,
        'rsi_oversold': strategies.rsi_oversold,
        'rsi_overbought': strategies.rsi_overbought,
        'volume_surge': strategies.volume_surge,
        'kd_golden_cross': strategies.kd_golden_cross,
        'macd_bullish': strategies.macd_bullish,
        'combined_bullish': strategies.combined_bullish,
    }

    if strategy_name not in strategy_methods:
        print(f"❌ 策略 '{strategy_name}' 不存在")
        return []

    strategy_func = strategy_methods[strategy_name]
    matched_stocks = []

    for stock_code, df in stock_data_dict.items():
        if df is not None and not df.empty:
            try:
                if strategy_func(df):
                    stock_name = stock_code
                    if stock_info_df is not None:
                        match = stock_info_df[stock_info_df['code'] == stock_code]
                        if not match.empty:
                            stock_name = f"{match.iloc[0]['name']} ({stock_code})"
                    matched_stocks.append(stock_name)
            except Exception as e:
                print(f"分析 {stock_code} 時發生錯誤: {e}")

    return matched_stocks


print("✅ TradingStrategies 策略定義完成！")
print("\n可用策略：")
print("  1. golden_cross - 黃金交叉")
print("  2. rsi_oversold - RSI 超賣")
print("  3. rsi_overbought - RSI 超買")
print("  4. volume_surge - 量增價漲")
print("  5. kd_golden_cross - KD 黃金交叉")
print("  6. macd_bullish - MACD 多頭")
print("  7. combined_bullish - 綜合多頭策略")

## 4. 初始化分析器

In [ ]:
# 建立分析器實例
analyzer = TaiwanStockAnalyzer()
print("✅ 分析器初始化完成！")

## 5. 獲取成交量前 50 名股票

In [ ]:
# 獲取成交量前 50 名股票
top_50_stocks = analyzer.get_top_volume_stocks(top_n=50)

# 顯示結果
print("\n📊 成交量前 50 名股票：\n")
display(top_50_stocks)

## 6. 分析這些股票的技術指標

In [ ]:
# 分析技術指標（這會需要一些時間）
stock_data = analyzer.analyze_stocks(top_50_stocks, period='3mo')

## 7. 顯示技術指標摘要

In [ ]:
# 顯示技術指標摘要
summary_df = analyzer.display_analysis_summary(stock_data, top_50_stocks)

## 8. 應用投資策略篩選股票

In [ ]:
# 應用不同策略
print("\n" + "="*60)
print("🎯 投資策略篩選結果")
print("="*60 + "\n")

strategies_to_test = [
    'golden_cross',
    'rsi_oversold',
    'volume_surge',
    'kd_golden_cross',
    'macd_bullish',
    'combined_bullish'
]

strategy_names_zh = {
    'golden_cross': '黃金交叉',
    'rsi_oversold': 'RSI 超賣',
    'volume_surge': '量增價漲',
    'kd_golden_cross': 'KD 黃金交叉',
    'macd_bullish': 'MACD 多頭',
    'combined_bullish': '綜合多頭策略'
}

for strategy in strategies_to_test:
    matched = apply_strategy(stock_data, strategy, top_50_stocks)
    print(f"\n📌 【{strategy_names_zh[strategy]}】 符合條件的股票 ({len(matched)} 支)：")
    if matched:
        for stock in matched:
            print(f"   ✅ {stock}")
    else:
        print("   ❌ 無符合條件的股票")

print("\n" + "="*60)

## 9. 繪製特定股票的技術分析圖表

In [ ]:
# 選擇一支股票繪製圖表（例如：台積電 2330.TW）
target_stock = '2330.TW'

if target_stock in stock_data:
    analyzer.plot_stock_chart(target_stock, stock_data[target_stock])
else:
    print(f"❌ {target_stock} 數據不存在")

## 10. 自訂策略範例

In [ ]:
# 定義自訂策略
def my_custom_strategy(df):
    """
    自訂策略範例：
    - RSI 介於 30-50（從超賣回升）
    - MA5 > MA20（短期向上）
    - 成交量 > 20日平均（量能放大）
    """
    if len(df) < 20:
        return False

    current = df.iloc[-1]
    avg_volume = df['Volume'].iloc[-20:].mean()

    return (
        30 < current['RSI'] < 50 and
        current['MA5'] > current['MA20'] and
        current['Volume'] > avg_volume
    )


# 應用自訂策略
print("\n🎯 自訂策略篩選結果\n")
custom_matched = []

for stock_code, df in stock_data.items():
    if df is not None and not df.empty:
        try:
            if my_custom_strategy(df):
                match = top_50_stocks[top_50_stocks['code'] == stock_code]
                if not match.empty:
                    stock_name = f"{match.iloc[0]['name']} ({stock_code})"
                    custom_matched.append(stock_name)
        except:
            pass

print(f"符合自訂策略的股票 ({len(custom_matched)} 支)：")
for stock in custom_matched:
    print(f"  ✅ {stock}")

if not custom_matched:
    print("  ❌ 無符合條件的股票")

## 11. 分析單一股票（快速測試）

In [ ]:
# 快速分析單一股票
def quick_analyze(stock_code):
    """快速分析單一股票"""
    print(f"\n🔍 快速分析 {stock_code}\n")

    # 獲取數據
    df = analyzer.fetch_stock_data(stock_code, period='3mo')

    if df is None:
        return

    # 計算技術指標
    df = analyzer.calculate_technical_indicators(df)

    # 顯示最新數據
    latest = df.iloc[-1]
    print(f"\n📈 最新數據（{latest.name.date()}）：")
    print(f"  收盤價: {latest['Close']:.2f}")
    print(f"  MA5: {latest['MA5']:.2f}")
    print(f"  MA20: {latest['MA20']:.2f}")
    print(f"  RSI: {latest['RSI']:.2f}")
    print(f"  K: {latest['K']:.2f}")
    print(f"  D: {latest['D']:.2f}")
    print(f"  成交量: {latest['Volume']:,.0f}")

    # 繪製圖表
    analyzer.plot_stock_chart(stock_code, df)


# 測試（請修改股票代碼）
quick_analyze('2330.TW')  # 台積電

## 💡 使用提示

1. **修改分析範圍**：將 `top_n=50` 改成其他數字以調整分析的股票數量
2. **調整時間範圍**：將 `period='3mo'` 改成 `'1mo'`、`'6mo'`、`'1y'` 等
3. **自訂策略**：在第 10 節中編寫自己的策略函數
4. **繪製圖表**：在第 9 節中修改 `target_stock` 以查看不同股票的圖表
5. **快速測試**：使用第 11 節的 `quick_analyze()` 函數快速分析單一股票

## ⚠️ 注意事項

- Yahoo Finance API 有請求限制，程式中已加入延遲避免被封鎖
- 技術分析僅供參考，投資有風險，請謹慎評估
- 建議使用至少 3 個月的歷史數據
- 台股代碼格式：股票代碼 + `.TW`（例如：2330.TW）

## 📚 延伸學習

- [yfinance 文件](https://pypi.org/project/yfinance/)
- [pandas-ta 文件](https://github.com/twopirllc/pandas-ta)
- [技術指標教學](https://www.investopedia.com/technical-analysis-4689657)